# Pipeline de Embeddings de Currículos Lattes

Este notebook orquestra o pipeline completo: extração em streaming do `.tar.zst`, chunking em markdown, embeddings com Qwen3-Embedding-4B (8-bit), checkpointing atômico e persistência em Parquet.

Toda a lógica pesada vive em módulos Python (`extract_archive.py`, `chunker.py`, `checkpoint.py`, `embedder.py`, `save_parquet.py`) — aqui apenas os importamos e usamos.

## CÉLULA 1 — Instalação de dependências

Assuma ambiente Colab com GPU T4 disponível. Instalamos todas as libs necessárias.

In [ ]:
!pip install -q sentence-transformers bitsandbytes transformers zstandard pyarrow torch

## CÉLULA 2 — Configuração

Todas as variáveis configuráveis do pipeline.

In [ ]:
ARCHIVE_PATH = "/content/curriculos.tar.zst"      # arquivo de entrada com ~180k currículos
OUTPUT_DIR = "/content/output_parquet"            # diretório onde os .parquet serão salvos
CHECKPOINT_PATH = "/content/checkpoint.json"      # arquivo de checkpoint (progresso)

MODEL_NAME = "Qwen/Qwen3-Embedding-4B"            # modelo de embedding
DEVICE = "cuda"                                   # T4 disponível

MAX_CHUNK_TOKENS = 500                            # tokens máximos por chunk
SHARD_SIZE = 1000                                 # nº de currículos por arquivo parquet
EMBED_BATCH_SIZE = 32                             # batch size do model.encode()


## CÉLULA 3 — Carregar modelo e tokenizer (1 única vez)

In [ ]:
from embedder import load_embedding_model, get_tokenizer

model = load_embedding_model(MODEL_NAME, device=DEVICE)
tokenizer = get_tokenizer(model)

print(f"Modelo carregado: {MODEL_NAME}")
print(f"Dimensão do embedding: {model.get_sentence_embedding_dimension()}")

## CÉLULA 4 — Carregar checkpoint

Saber quais `lattes_id` já foram processados permite retomar do ponto em que parou caso o Colab caia.

In [ ]:
from checkpoint import load_checkpoint

processed_ids = load_checkpoint(CHECKPOINT_PATH)
print(f"Currículos já processados no checkpoint: {len(processed_ids)}")

## CÉLULA 5 — Loop principal do pipeline

1. Iterar `iter_curriculum_files`.
2. Pular currículos já no checkpoint.
3. Acumular até `SHARD_SIZE` currículos não processados.
4. Para cada shard: chunking, um único `embed_texts` para o shard inteiro, salvar parquet e atualizar checkpoint.
5. Tratar cada currículo individual com try/except (arquivo ruim não derruba o pipeline).

In [ ]:
import gc
import logging
import os
import time
from datetime import datetime, timezone

import torch
from tqdm.auto import tqdm

from extract_archive import iter_curriculum_files
from chunker import chunk_markdown
from embedder import embed_texts
from save_parquet import save_shard_parquet
from checkpoint import save_checkpoint

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("pipeline")

os.makedirs(OUTPUT_DIR, exist_ok=True)


def release_memory():
    """Libera buffers da GPU entre shards para evitar fragmentação acumulada."""
    gc.collect()
    torch.cuda.empty_cache()


def process_shard(shard_curricula, shard_index):
    """Chunk + embed + salvar um shard inteiro de uma vez."""
    shard_chunks = []
    metadata = {}  # lattes_id -> (filename, timestamp)

    # (a) Chunking por currículo, com chunk_index sequencial por lattes_id
    for record in shard_curricula:
        lattes_id = record["lattes_id"]
        filename = record["filename"]
        content = record["content"]
        chunks = chunk_markdown(content, tokenizer, max_tokens=MAX_CHUNK_TOKENS)
        for idx, chunk in enumerate(chunks):
            shard_chunks.append({
                "lattes_id": lattes_id,
                "chunk_index": idx,
                "content": chunk["content"],
                "secao": chunk["secao"],
            })
        metadata[lattes_id] = filename

    if not shard_chunks:
        return shard_curricula, 0

    # (b) Embed do shard inteiro de uma vez (maximiza batch/throughput da GPU)
    texts = [c["content"] for c in shard_chunks]
    batch_meta = [
        f"{metadata[c['lattes_id']]} (lattes_id={c['lattes_id']}, chunk={c['chunk_index']})"
        for c in shard_chunks
    ]
    embeddings, skipped_indices = embed_texts(
        model, texts, batch_size=EMBED_BATCH_SIZE, batch_meta=batch_meta
    )

    # (c) Montar records finais e salvar (sem os chunks que deram OOM)
    skipped_set = set(skipped_indices)
    kept_chunks = [
        c for i, c in enumerate(shard_chunks) if i not in skipped_set
    ]
    timestamp = datetime.now(timezone.utc).isoformat()
    records = []
    for i, chunk in enumerate(kept_chunks):
        records.append({
            "lattes_id": chunk["lattes_id"],
            "chunk_index": chunk["chunk_index"],
            "content": chunk["content"],
            "metadata_filename": metadata[chunk["lattes_id"]],
            "metadata_timestamp": timestamp,
            "embedding": embeddings[i],
        })

    output_path = os.path.join(OUTPUT_DIR, f"shard_{shard_index}.parquet")
    save_shard_parquet(records, output_path)
    return shard_curricula, len(records)


start_time = time.time()
total_chunks = 0
total_processed = len(processed_ids)
shard_index = 0
skipped = 0

shard_curricula = []
progress_bar = tqdm(total=None, desc="Currículos processados")
progress_bar.update(total_processed)

for record in iter_curriculum_files(ARCHIVE_PATH):
    lattes_id = record["lattes_id"]

    # (2) Pular currículos já processados
    if lattes_id in processed_ids:
        skipped += 1
        continue

    # (5) Currículo individual em try/except — arquivo ruim não derruba o pipeline
    try:
        chunk_markdown(record["content"], tokenizer, max_tokens=MAX_CHUNK_TOKENS)
    except Exception as e:
        logger.warning("Erro ao parsear currículo %s: %s", lattes_id, e)
        continue

    shard_curricula.append(record)

    # (3) Quando o shard estiver cheio, processá-lo
    if len(shard_curricula) >= SHARD_SIZE:
        shard_ids, shard_chunk_count = process_shard(shard_curricula, shard_index)
        shard_index += 1
        total_chunks += shard_chunk_count

        # (4d) Atualizar checkpoint imediatamente após salvar o shard
        processed_ids.update(rid["lattes_id"] for rid in shard_ids)
        total_processed = len(processed_ids)
        save_checkpoint(CHECKPOINT_PATH, processed_ids)

        # Limpeza de memória ao fim de cada shard
        release_memory()

        elapsed = time.time() - start_time
        print(
            f"Shard {shard_index} salvo | "
            f"currículos processados: {total_processed} | "
            f"chunks totais: {total_chunks} | "
            f"tempo decorrido: {elapsed:.1f}s"
        )
        progress_bar.update(len(shard_ids))
        shard_curricula = []

# Processar o resto que ficou no buffer (shard parcial)
if shard_curricula:
    shard_ids, shard_chunk_count = process_shard(shard_curricula, shard_index)
    shard_index += 1
    total_chunks += shard_chunk_count
    processed_ids.update(rid["lattes_id"] for rid in shard_ids)
    total_processed = len(processed_ids)
    save_checkpoint(CHECKPOINT_PATH, processed_ids)

    # Limpeza de memória ao fim do shard parcial
    release_memory()

    elapsed = time.time() - start_time
    print(
        f"Shard final {shard_index} salvo | "
        f"currículos processados: {total_processed} | "
        f"chunks totais: {total_chunks} | "
        f"tempo decorrido: {elapsed:.1f}s"
    )
    progress_bar.update(len(shard_ids))

progress_bar.close()

elapsed_total = time.time() - start_time
print("\nPipeline concluído!")
print(f"  Currículos processados no total: {total_processed}")
print(f"  Currículos pulados (já no checkpoint): {skipped}")
print(f"  Chunks gerados no total: {total_chunks}")
print(f"  Shards salvos: {shard_index}")
print(f"  Tempo total: {elapsed_total:.1f}s")

## CÉLULA 7 — Validação rápida (opcional, comentada)

Carregar um parquet gerado para conferir schema e shape manualmente.

In [ ]:
# import pandas as pd
# import glob

# parquet_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "shard_*.parquet")))
# if parquet_files:
#     df = pd.read_parquet(parquet_files[0])
#     print(f"Arquivo: {parquet_files[0]}")
#     print(f"Shape: {df.shape}")
#     print(df.head())
#     print("\nDtypes:")
#     print(df.dtypes)
# else:
#     print("Nenhum arquivo parquet encontrado em", OUTPUT_DIR)